In [1]:
import random
from typing import List, Tuple
import numpy as np

random.seed(42)
np.random.seed(42)


class BusRouteOptimizer:

    def __init__(self, distance_matrix: np.ndarray, pop_size: int = 50, elite_size: int = 5, mutation_rate: float = 0.02, generations: int = 200):
        self.distance_matrix = distance_matrix
        self.num_stops = len(distance_matrix)
        self.pop_size = pop_size
        self.elite_size = elite_size
        self.mutation_rate = mutation_rate
        self.generations = generations

    def calculate_route_distance(self, route: List[int]) -> float:
        return sum(self.distance_matrix[route[i]][route[(i + 1) % len(route)]] for i in range(len(route)))

    def calculate_fitness(self, route: List[int]) -> float:
        distance = self.calculate_route_distance(route)
        return 1.0 / distance if distance != 0 else 0.0

    def create_initial_population(self) -> List[List[int]]:
        stops_to_visit = list(range(1, self.num_stops))
        return [[0] + random.sample(stops_to_visit, len(stops_to_visit)) for _ in range(self.pop_size)]

    def rank_population(self, population: List[List[int]]) -> List[Tuple[int, float]]:
        fitness_scores = [(i, self.calculate_fitness(route)) for i, route in enumerate(population)]
        return sorted(fitness_scores, key=lambda x: x[1], reverse=True)

    def selection(self, pop_ranked: List[Tuple[int, float]]) -> List[int]:
        selection_results = [pop_ranked[i][0] for i in range(self.elite_size)]

        fitness_scores = [item[1] for item in pop_ranked]
        total_fitness = sum(fitness_scores)
        probabilities = [f / total_fitness for f in fitness_scores]

        rem_size = self.pop_size - self.elite_size
        chosen_indices = np.random.choice(len(pop_ranked), size=rem_size, p=probabilities)
        selection_results.extend([pop_ranked[idx][0] for idx in chosen_indices])

        return selection_results

    def ordered_crossover(self, parent1: List[int], parent2: List[int]) -> List[int]:
        p1_body, p2_body = parent1[1:], parent2[1:]

        gene_a, gene_b = int(random.random() * len(p1_body)), int(random.random() * len(p1_body))
        start_gene, end_gene = min(gene_a, gene_b), max(gene_a, gene_b)

        child_part1 = p1_body[start_gene:end_gene]
        child_part2 = [item for item in p2_body if item not in child_part1]

        return [0] + child_part1 + child_part2

    def breed_population(self, population: List[List[int]], selection_results: List[int]) -> List[List[int]]:
        next_generation = [population[selection_results[i]] for i in range(self.elite_size)]
        pool = [population[idx] for idx in selection_results]

        for i in range(self.pop_size - self.elite_size):
            parent1 = pool[i]
            parent2 = pool[len(pool) - i - 1]
            next_generation.append(self.ordered_crossover(parent1, parent2))

        return next_generation

    def mutate(self, route: List[int]) -> List[int]:
        for swapped_idx in range(1, len(route)):
            if random.random() < self.mutation_rate:
                target_idx = random.randint(1, len(route) - 1)
                route[swapped_idx], route[target_idx] = route[target_idx], route[swapped_idx]
        return route

    def mutate_population(self, population: List[List[int]]) -> List[List[int]]:
        return [self.mutate(route) if i >= self.elite_size else route for i, route in enumerate(population)]

    def optimize(self) -> Tuple[List[int], float]:
        population = self.create_initial_population()
        print(f"Initial best distance: {1 / self.rank_population(population)[0][1]:.2f}")

        for generation in range(self.generations):
            population = self.mutate_population(self.breed_population(population, self.selection(self.rank_population(population))))
            if (generation + 1) % 50 == 0:
                print(f"Generation {generation + 1} | Best distance: {1 / self.rank_population(population)[0][1]:.2f}")

        best_route = population[self.rank_population(population)[0][0]]
        return best_route, self.calculate_route_distance(best_route)


if __name__ == "__main__":
    mock_distance_matrix = np.array([
        [0, 10, 15, 20, 25, 30],
        [10, 0, 35, 25, 15, 12],
        [15, 35, 0, 30, 10, 22],
        [20, 25, 30, 0, 20, 17],
        [25, 15, 10, 20, 0, 14],
        [30, 12, 22, 17, 14, 0]
    ])

    optimizer = BusRouteOptimizer(
        distance_matrix=mock_distance_matrix,
        pop_size=20, elite_size=2, mutation_rate=0.05, generations=100
    )

    best_bus_route, optimized_distance = optimizer.optimize()

    route_string = " -> ".join(f"Stop {stop}" for stop in best_bus_route + [best_bus_route[0]])

    print(f"\n--- Optimization Complete ---\nOptimized Route: {route_string}\nTotal Distance:  {optimized_distance} km")

Initial best distance: 104.00
Generation 50 | Best distance: 84.00
Generation 100 | Best distance: 84.00

--- Optimization Complete ---
Optimized Route: Stop 0 -> Stop 1 -> Stop 5 -> Stop 3 -> Stop 4 -> Stop 2 -> Stop 0
Total Distance:  84 km
